In [1]:
paths = {
    "W": {"distance": 300, "time": 25, "congestion": "Medium"},
    "X": {"distance": 200, "time": 15, "congestion": "Low"},
    "Y": {"distance": 150, "time": 20, "congestion": "Medium"},
    "Z": {"distance": 250, "time": 10, "congestion": "High"}
}

congestion_score = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

def utility(path):
    return -(path["distance"] * 0.4 +
             path["time"] * 0.4 +
             congestion_score[path["congestion"]] * 0.2)

utilities = {}

for name, data in paths.items():
    utilities[name] = utility(data)

for p, u in utilities.items():
    print(f"Path {p} Utility: {u}")

best_path = max(utilities, key=utilities.get)
print("\nSelected Path:", best_path)


Path W Utility: -130.4
Path X Utility: -86.2
Path Y Utility: -68.4
Path Z Utility: -104.6

Selected Path: Y


In [ ]:
import random

class GreenhouseSection:
    def __init__(self, name, initial_state="Healthy"):
        self.name = name
        self.actual_state = initial_state  # Hidden from agent
        self.agent_belief = initial_state   # Agent's belief
        self.days_since_watered = 0
        self.water_needed_threshold = 2
    
    def update_actual_state(self):
        """Update actual plant state based on watering"""
        self.days_since_watered += 1
        if self.days_since_watered >= self.water_needed_threshold:
            self.actual_state = "Wilted"
        # Healthy if recently watered
        elif self.days_since_watered == 0:
            self.actual_state = "Healthy"
    
    def water(self):
        """Water the plants in this section"""
        self.days_since_watered = 0
        self.actual_state = "Healthy"
    
    def observe(self):
        """Agent observes this section (with 80% accuracy)"""
        # Simulate partial observability with some error
        if random.random() < 0.8:  # 80% accurate observation
            return self.actual_state
        else:
            return "Healthy" if self.actual_state == "Wilted" else "Wilted"
    
    def update_belief(self, observation):
        """Update agent's belief based on observation"""
        self.agent_belief = observation

class ModelBasedPlantCareAgent:
    def __init__(self):
        self.current_location = "Section1"
        self.internal_model = {
            "Section1": {"belief": "Healthy", "last_watered": 0, "confidence": 1.0},
            "Section2": {"belief": "Healthy", "last_watered": 0, "confidence": 0.5}
        }
        self.memory = []  # Track past actions and observations
        self.water_used = 0
    
    def update_internal_model(self, section_name, observation):
        """Update internal model based on new observation"""
        self.internal_model[section_name]["belief"] = observation
        
        # Increase confidence when we directly observe
        if section_name == self.current_location:
            self.internal_model[section_name]["confidence"] = min(1.0, 
                self.internal_model[section_name]["confidence"] + 0.3)
        else:
            # Decrease confidence for unobserved sections
            self.internal_model[section_name]["confidence"] = max(0.1,
                self.internal_model[section_name]["confidence"] - 0.1)
        
        # Track in memory
        self.memory.append({
            "time": len(self.memory),
            "location": self.current_location,
            "observation": observation,
            "section": section_name
        })
    
    def decide_action(self):
        """Decide next action based on internal model"""
        current_section = self.current_location
        other_section = "Section2" if current_section == "Section1" else "Section1"
        
        # Check current section
        if self.internal_model[current_section]["belief"] == "Wilted":
            return "water"
        
        # Check other section if we're more confident it needs water
        current_confidence = self.internal_model[current_section]["confidence"]
        other_confidence = self.internal_model[other_section]["confidence"]
        
        if (self.internal_model[other_section]["belief"] == "Wilted" and 
            other_confidence > 0.7):
            return "move"
        
        # If we've been here too long, check other section
        if len([m for m in self.memory[-3:] if m["location"] == current_section]) >= 2:
            return "move"
        
        return "water" if random.random() < 0.3 else "wait"
    
    def execute_action(self, action, greenhouse):
        """Execute chosen action"""
        if action == "move":
            self.current_location = "Section2" if self.current_location == "Section1" else "Section1"
            return f"Moved to {self.current_location}"
        
        elif action == "water":
            section = greenhouse[self.current_location]
            section.water()
            self.internal_model[self.current_location]["last_watered"] = 0
            self.internal_model[self.current_location]["belief"] = "Healthy"
            self.water_used += 1
            return f"Watered {self.current_location}"
        
        return "Waiting and observing"
    
    def display_state(self, greenhouse, step):
        """Display current state"""
        print(f"\n{'='*60}")
        print(f"TIME STEP {step}")
        print(f"{'='*60}")
        print(f"Agent Location: {self.current_location}")
        print(f"Water Used Today: {self.water_used}")
        print("\nINTERNAL MODEL (Agent's Beliefs):")
        print(f"{'Section':<10} {'Belief':<10} {'Confidence':<12} {'Last Watered':<12}")
        for section, data in self.internal_model.items():
            print(f"{section:<10} {data['belief']:<10} {data['confidence']:<12.2f} {data['last_watered']:<12}")
        
        print("\nACTUAL STATE (Hidden from Agent):")
        print(f"{'Section':<10} {'Actual State':<12} {'Days Dry':<10}")
        for name, section in greenhouse.items():
            print(f"{name:<10} {section.actual_state:<12} {section.days_since_watered:<10}")
        
        print("\nOBSERVATIONS (Agent's Perception):")
        for name, section in greenhouse.items():
            observation = section.observe() if name == self.current_location else "Not visible"
            print(f"{name}: {observation}")

def simulate_greenhouse(days=10):
    """Run simulation"""
    # Initialize greenhouse
    greenhouse = {
        "Section1": GreenhouseSection("Section1", "Healthy"),
        "Section2": GreenhouseSection("Section2", "Healthy")
    }
    
    # Initialize agent
    agent = ModelBasedPlantCareAgent()
    
    print("INITIAL STATE:")
    print("Both sections start as Healthy")
    print("Agent starts in Section1")
    
    for day in range(1, days + 1):
        # Update actual plant states
        for section in greenhouse.values():
            section.update_actual_state()
        
        # Agent observes current location
        current_section = greenhouse[agent.current_location]
        observation = current_section.observe()
        
        # Update agent's internal model
        agent.update_internal_model(agent.current_location, observation)
        
        # Increment last_watered counters in model
        for section_name in agent.internal_model:
            agent.internal_model[section_name]["last_watered"] += 1
        
        # Decide and execute action
        action = agent.decide_action()
        action_result = agent.execute_action(action, greenhouse)
        
        # Display state
        agent.display_state(greenhouse, day)
        print(f"\nAgent Action: {action_result}")
        print(f"Decision Rationale: {get_decision_rationale(agent, action)}")
        
        # Small delay for readability
        if day < days:
            input("\nPress Enter for next time step...")

def get_decision_rationale(agent, action):
    """Explain why agent chose this action"""
    current = agent.current_location
    other = "Section2" if current == "Section1" else "Section1"
    
    if action == "water":
        return f"Believes {current} needs water (belief: {agent.internal_model[current]['belief']})"
    elif action == "move":
        return f"Moving to check {other} (confidence in {current}: {agent.internal_model[current]['confidence']:.2f})"
    else:
        return f"Monitoring {current}, no immediate action needed"

# Run simulation
if __name__ == "__main__":
    print("MODEL-BASED AGENT WITH INTERNAL STATE TRACKING")
    print("Greenhouse Plant Care Simulation")
    print("="*60)
    
    # Set random seed for reproducibility
    random.seed(42)
    
    simulate_greenhouse(days=8)

MODEL-BASED AGENT WITH INTERNAL STATE TRACKING
Greenhouse Plant Care Simulation
INITIAL STATE:
Both sections start as Healthy
Agent starts in Section1

TIME STEP 1
Agent Location: Section1
Water Used Today: 1

INTERNAL MODEL (Agent's Beliefs):
Section    Belief     Confidence   Last Watered
Section1   Healthy    1.00         0           
Section2   Healthy    0.50         1           

ACTUAL STATE (Hidden from Agent):
Section    Actual State Days Dry  
Section1   Healthy      0         
Section2   Healthy      1         

OBSERVATIONS (Agent's Perception):
Section1: Healthy
Section2: Not visible

Agent Action: Watered Section1
Decision Rationale: Believes Section1 needs water (belief: Healthy)



Press Enter for next time step... 



TIME STEP 2
Agent Location: Section2
Water Used Today: 1

INTERNAL MODEL (Agent's Beliefs):
Section    Belief     Confidence   Last Watered
Section1   Healthy    1.00         1           
Section2   Healthy    0.50         2           

ACTUAL STATE (Hidden from Agent):
Section    Actual State Days Dry  
Section1   Healthy      1         
Section2   Wilted       2         

OBSERVATIONS (Agent's Perception):
Section1: Not visible
Section2: Wilted

Agent Action: Moved to Section2
Decision Rationale: Moving to check Section1 (confidence in Section2: 0.50)
